# 01 — Data Cleaning

**Last run:** see notebook timestamp; cleaned dataset → `data/acled_clean.parquet`.

## Scope

This notebook produces the analysis-ready dataset that feeds `02_exploration.ipynb` and `03_analysis.ipynb`.

Inputs:
1. **`data/acled_raw.csv`** — ACLED Data Export Tool, Western Africa, 2018-01-01 → 2025-04-25, all event types. 69,598 rows × 31 cols.
2. **`data/powell_thyne_coups.csv`** — Powell-Thyne global coup attempts dataset, V2026.01.13.
3. **`data/vdem_regime.csv`** — V-Dem v16 (Mar 2026) country-year subset for Mali, Burkina Faso, Niger, 2018–2025.

Output:
- **`data/acled_clean.parquet`** — cleaned ACLED events for the AES core (Mali, Burkina Faso, Niger), 2018-01-01 → 2025-04-25, with derived columns and merged Powell-Thyne + V-Dem features.
- **`data/acled_clean_west_africa.parquet`** — same cleaning applied to the full Western-Africa export (16 countries) for the optional comparative analyses.

## Data-generating process (documented per the rubric)

ACLED is a real-time event dataset compiled by trained researchers from a hierarchy of sources: international newswires (Reuters, AFP), national newspapers, regional outlets, NGO and humanitarian situation reports, social-media-verified accounts, and (from 2018 onward) partnerships with conflict observatories (e.g., Centre for Humanitarian Dialogue). Each row encodes a discrete politically violent or politically contentious event with date, location (geocoded to admin1/admin2), event type, sub-event type, actor1/actor2, fatalities, and a free-text `notes` field with the source narrative. The full coding methodology is documented in Raleigh et al. 2010 *Journal of Peace Research* (introducing ACLED) and the ACLED Codebook.

## Known biases and how we handle them

1. **Source-availability bias.** Areas with denser media coverage produce more recorded events; remote rural areas (much of Sahelian Mali/Burkina Faso/Niger) are under-coverage at the margin. Eck 2012 *Cooperation and Conflict* documents this critique. **Our handling:** explicit caveat in the report's Discussion; cross-validate against UCDP-GED for spot-checks where feasible.
2. **Fatality conservatism.** ACLED reports the most conservative estimate when sources disagree. The reported `fatalities` column therefore systematically understates. **Our handling:** all fatality counts are described as 'reported fatalities,' not 'actual deaths.'
3. **Actor-coding evolution.** ACLED has revised actor classifications over time, particularly for jihadist and externally-supplied groups (e.g., 'Wagner Group (Russia)' → 'Africa Corps (Russia)' rebranding from 2024 onward). **Our handling:** in §1.4 below we build a normalised `actor_role` classification (`state`, `non_state_armed_group`, `external_force`, `civilian`, `other`) using the `inter1` interaction code, which is more stable than free-text actor names.

## Retrieval process (filters used)

| Filter | Value |
|---|---|
| Date range | 2018-01-01 to 2025-04-25 |
| Region | Western Africa |
| Countries | (all 16 Western African countries — region filter overrides country chips) |
| Event types | All 6 (Battles, Explosions/Remote violence, Violence against civilians, Protests, Riots, Strategic developments) |
| Actor / sub-event / interaction filters | none (apply downstream in pandas) |
| Output options | dyadic (one row per event), text interaction codes, comma-delimited CSV |
| Population fields | excluded (not used in this analysis) |


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

DATA = Path('../data').resolve()
AES = ['Mali', 'Burkina Faso', 'Niger']
WINDOW_START = pd.Timestamp('2018-01-01')
WINDOW_END = pd.Timestamp('2025-04-25')

print(f'Data folder: {DATA}')
print(f'AES core   : {AES}')
print(f'Window     : {WINDOW_START.date()} → {WINDOW_END.date()}')

Data folder: /Users/rlatldn20001114/Desktop/poli3148 assignment 1/data
AES core   : ['Mali', 'Burkina Faso', 'Niger']
Window     : 2018-01-01 → 2025-04-25


## 1.1  Load ACLED raw and run integrity assertions

In [2]:
raw = pd.read_csv(DATA/'acled_raw.csv', low_memory=False)
print(f'Raw shape: {raw.shape}')

# Integrity assertions: if any of these fail, re-run the export.
assert raw['event_date'].min() == '2018-01-01', f"Min date is {raw['event_date'].min()}, expected 2018-01-01"
assert raw['event_date'].max() == '2025-04-25', f"Max date is {raw['event_date'].max()}, expected 2025-04-25"
assert set(AES).issubset(set(raw['country'].unique())), 'Missing one of the AES countries'
assert raw['event_type'].nunique() == 6, 'Expected 6 ACLED event types'

print('All integrity assertions passed.')
print(f'\nCountries in raw export ({raw.country.nunique()}):')
print(raw['country'].value_counts().to_string())

Raw shape: (69598, 31)
All integrity assertions passed.

Countries in raw export (16):
country
Nigeria          30404
Burkina Faso     11528
Mali             11029
Niger             4420
Guinea            2194
Ghana             2163
Mauritania        1675
Benin             1618
Ivory Coast       1499
Senegal           1198
Liberia            488
Sierra Leone       485
Togo               338
Cape Verde         278
Guinea-Bissau      163
Gambia             118


## 1.2  Type cleaning + duplicate check (applied to the full West-Africa frame)

In [3]:
df = raw.copy()

# Datetime conversion
df['event_date'] = pd.to_datetime(df['event_date'])
df['year'] = df['event_date'].dt.year.astype(int)
df['month'] = df['event_date'].dt.month.astype(int)
df['year_month'] = df['event_date'].dt.to_period('M').dt.to_timestamp()
df['quarter'] = df['event_date'].dt.to_period('Q').dt.to_timestamp()

# Numeric cleanups
df['fatalities'] = pd.to_numeric(df['fatalities'], errors='coerce').fillna(0).astype(int)
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df['geo_precision'] = pd.to_numeric(df['geo_precision'], errors='coerce')

# String trimming on key categoricals
for c in ['country','region','admin1','admin2','event_type','sub_event_type',
          'actor1','actor2','assoc_actor_1','assoc_actor_2','inter1','inter2',
          'civilian_targeting','disorder_type','source_scale']:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip().replace({'nan': pd.NA})

# Duplicates on event_id_cnty (ACLED's primary key)
n_dup = df.duplicated(subset=['event_id_cnty']).sum()
assert n_dup == 0, f'{n_dup} duplicate event_id_cnty rows found'

# Lat/lon sanity
assert df['latitude'].between(-90, 90).all(), 'Latitude out of range'
assert df['longitude'].between(-180, 180).all(), 'Longitude out of range'

print(f'Clean West-Africa frame: {df.shape}')
print(f'Duplicates on event_id_cnty: {n_dup}')
print(f'Total reported fatalities (West Africa, full window): {df.fatalities.sum():,}')

Clean West-Africa frame: (69598, 34)
Duplicates on event_id_cnty: 0
Total reported fatalities (West Africa, full window): 129,677


## 1.3  Derived column — `actor_role`

ACLED's `inter1` and `inter2` columns are the most stable actor-classification fields (they survive ACLED's periodic actor-name rebrandings). We collapse the eight ACLED interaction categories into five analytical categories:

| ACLED `inter1` value | Our `actor_role` |
|---|---|
| State Forces | `state` |
| Rebel Group, Political Militia | `non_state_armed_group` |
| Identity Militia, Communal Group | `non_state_armed_group` |
| External/Other Forces | `external_force` |
| Civilians | `civilian` |
| Protesters, Rioters | `civilian` |
| (anything else) | `other` |

We also build a per-event indicator `civilian_targeted` that is True when ACLED's `civilian_targeting` field equals `'Civilian targeting'`.

In [4]:
INTER_TO_ROLE = {
    'state forces': 'state',
    'rebel group': 'non_state_armed_group',
    'political militia': 'non_state_armed_group',
    'identity militia': 'non_state_armed_group',
    'communal group': 'non_state_armed_group',
    'external/other forces': 'external_force',
    'civilians': 'civilian',
    'protesters': 'civilian',
    'rioters': 'civilian',
    'other': 'other',
}

def map_role(v):
    if pd.isna(v):
        return 'other'
    return INTER_TO_ROLE.get(str(v).strip().lower(), 'other')

df['actor1_role'] = df['inter1'].apply(map_role)
df['actor2_role'] = df['inter2'].apply(map_role)
df['civilian_targeted'] = (df['civilian_targeting'].fillna('').str.lower() == 'civilian targeting')

print('actor1_role distribution (West Africa, full window):')
print(df['actor1_role'].value_counts().to_string())
print()
print(f"Civilian-targeted events: {df['civilian_targeted'].sum():,}")
print(f"Civilian-targeted fatalities: {df.loc[df['civilian_targeted'], 'fatalities'].sum():,}")

actor1_role distribution (West Africa, full window):
actor1_role
non_state_armed_group    37952
civilian                 19335
state                    11427
external_force             884

Civilian-targeted events: 24,694
Civilian-targeted fatalities: 48,868


## 1.4  Derived column — `external_force_present`

Captures whether either actor is an external state/PMC force (French Barkhane / MINUSMA / Wagner Group / Africa Corps / Russian state). This is the central treatment variable for our research question.

We use a regex match on `actor1`, `actor2`, `assoc_actor_1`, `assoc_actor_2` to flag external-force involvement, and a finer flag for which external force.

In [5]:
EXTERNAL_PATTERNS = {
    'french': r'(?:France|French|Barkhane|Serval|Sabre)',
    'wagner': r'(?:Wagner|Africa Corps|Russia|Russian)',
    'un': r'(?:MINUSMA|United Nations|UN Mission)',
    'us': r'(?:United States|US Army|U\.S\. Forces)',
    'ecowas': r'(?:ECOWAS|G5 Sahel)',
}

actor_cols = ['actor1','assoc_actor_1','actor2','assoc_actor_2']
actors_concat = df[actor_cols].fillna('').agg(' | '.join, axis=1)

for label, pat in EXTERNAL_PATTERNS.items():
    df[f'ext_{label}'] = actors_concat.str.contains(pat, case=False, regex=True, na=False)

df['external_force_present'] = df[[f'ext_{k}' for k in EXTERNAL_PATTERNS]].any(axis=1)

print('External-force involvement counts (West Africa, full window):')
for k in EXTERNAL_PATTERNS:
    print(f'  {k:8s}: {df[f"ext_{k}"].sum():,} events')
print(f"ANY      : {df['external_force_present'].sum():,} events")

External-force involvement counts (West Africa, full window):
  french  : 466 events
  wagner  : 1,325 events
  un      : 417 events
  us      : 26 events
  ecowas  : 39 events
ANY      : 2,223 events


## 1.5  AES-core slice (Mali, Burkina Faso, Niger)

All downstream analyses run on this AES slice. The Western-Africa frame is preserved separately for any optional comparative analyses.

In [6]:
aes = df[df['country'].isin(AES)].copy()
print(f'AES-core events: {len(aes):,}  (vs West-Africa total: {len(df):,})')
print()
print('AES year × country event counts:')
print(pd.crosstab(aes['year'], aes['country']).to_string())
print()
print(f'AES total reported fatalities: {aes.fatalities.sum():,}')

AES-core events: 26,977  (vs West-Africa total: 69,598)

AES year × country event counts:
country  Burkina Faso  Mali  Niger
year                              
2018              402   751    168
2019              892   822    371
2020              877  1268    550
2021             1868  1355    431
2022             2756  1904    989
2023             2360  2250    798
2024             1747  1992    786
2025              626   687    327

AES total reported fatalities: 61,007


## 1.6  Powell-Thyne coup-event timeline (treatment-date markers)

Powell-Thyne codes `coup` as 1 (failed) or 2 (successful). We retain both for the AES window and align the date columns.

In [7]:
pt = pd.read_csv(DATA/'powell_thyne_coups.csv')
pt = pt[pt['country'].isin(AES) & (pt['year']>=2018) & (pt['year']<=2025)].copy()
pt['coup_date'] = pd.to_datetime(
    dict(year=pt['year'], month=pt['month'], day=pt['day'])
)
pt['coup_outcome'] = pt['coup'].map({1:'failed', 2:'successful'})
pt = pt[['country','coup_date','coup_outcome','version']].sort_values(['country','coup_date'])
pt.to_parquet(DATA/'aes_coups.parquet', index=False)
print('AES coup events (Powell-Thyne V2026.01.13):')
print(pt.to_string(index=False))

AES coup events (Powell-Thyne V2026.01.13):
     country  coup_date coup_outcome     version
Burkina Faso 2022-01-23   successful V2026.01.13
Burkina Faso 2022-09-30   successful V2026.01.13
        Mali 2020-08-18   successful V2026.01.13
        Mali 2021-05-21   successful V2026.01.13
       Niger 2021-03-31       failed V2026.01.13
       Niger 2023-07-26   successful V2026.01.13


## 1.7  External-force timeline (hand-coded from verified open-source reporting)

These dates anchor the report's mechanism analysis. Each date was verified by `search-specialist` in Phase 0 against published reporting (Reuters, AFP, Le Monde, ICG, ACLED situation reports). They mark substitution events between French Barkhane / MINUSMA and Wagner Group / Africa Corps.

In [8]:
ext_timeline = pd.DataFrame([
    {'country':'Mali',         'event':'Wagner deployment begins',       'date':'2021-12-01'},
    {'country':'Mali',         'event':'French Barkhane formally ends',   'date':'2022-11-09'},
    {'country':'Mali',         'event':'MINUSMA full withdrawal',         'date':'2023-12-31'},
    {'country':'Burkina Faso', 'event':'French withdrawal',               'date':'2023-02-23'},
    {'country':'Burkina Faso', 'event':'Africa Corps deployment begins',  'date':'2024-01-24'},
    {'country':'Niger',        'event':'French withdrawal complete',      'date':'2023-12-22'},
    {'country':'Niger',        'event':'Africa Corps deployment begins',  'date':'2024-04-11'},
])
ext_timeline['date'] = pd.to_datetime(ext_timeline['date'])
ext_timeline.to_parquet(DATA/'aes_external_timeline.parquet', index=False)
print(ext_timeline.to_string(index=False))

     country                          event       date
        Mali       Wagner deployment begins 2021-12-01
        Mali  French Barkhane formally ends 2022-11-09
        Mali        MINUSMA full withdrawal 2023-12-31
Burkina Faso              French withdrawal 2023-02-23
Burkina Faso Africa Corps deployment begins 2024-01-24
       Niger     French withdrawal complete 2023-12-22
       Niger Africa Corps deployment begins 2024-04-11


## 1.8  Merge V-Dem regime scores on country-year

In [9]:
vdem = pd.read_csv(DATA/'vdem_regime.csv')
vdem = vdem.rename(columns={'country_name':'country'})
vdem['year'] = vdem['year'].astype(int)

keep_v = ['country','year','v2x_polyarchy','v2x_libdem','v2x_regime','v2x_regime_amb','v2xcl_rol','v2x_civlib']
have = [c for c in keep_v if c in vdem.columns]
vdem_sub = vdem[have]

aes = aes.merge(vdem_sub, on=['country','year'], how='left')
df  = df.merge(vdem_sub, on=['country','year'], how='left')

miss = aes[aes['v2x_polyarchy'].isna()][['country','year']].drop_duplicates()
if len(miss):
    print('WARNING — country-years missing V-Dem (likely 2025 partial-year):')
    print(miss.to_string(index=False))
else:
    print('V-Dem merge complete — every AES country-year has a regime score.')

V-Dem merge complete — every AES country-year has a regime score.


## 1.9  Add post-coup / post-Wagner indicator columns to the AES frame

These three boolean columns are the workhorse for the report's pre/post comparisons.

| Column | Definition |
|---|---|
| `post_first_coup` | True if the event date is on or after the first successful coup in that country (Mali 2020-08-18; BF 2022-01-23; Niger 2023-07-26) |
| `post_french_exit` | True if the event date is on or after French Barkhane formally ended in that country |
| `post_russian_arrival` | True if the event date is on or after Wagner / Africa Corps formally deployed in that country |

In [10]:
FIRST_COUP = {
    'Mali':         pd.Timestamp('2020-08-18'),
    'Burkina Faso': pd.Timestamp('2022-01-23'),
    'Niger':        pd.Timestamp('2023-07-26'),
}
FRENCH_EXIT = {
    'Mali':         pd.Timestamp('2022-11-09'),
    'Burkina Faso': pd.Timestamp('2023-02-23'),
    'Niger':        pd.Timestamp('2023-12-22'),
}
RUSSIAN_ARRIVAL = {
    'Mali':         pd.Timestamp('2021-12-01'),
    'Burkina Faso': pd.Timestamp('2024-01-24'),
    'Niger':        pd.Timestamp('2024-04-11'),
}

aes['post_first_coup']      = aes.apply(lambda r: r['event_date'] >= FIRST_COUP[r['country']], axis=1)
aes['post_french_exit']     = aes.apply(lambda r: r['event_date'] >= FRENCH_EXIT[r['country']], axis=1)
aes['post_russian_arrival'] = aes.apply(lambda r: r['event_date'] >= RUSSIAN_ARRIVAL[r['country']], axis=1)

print('Pre/post breakpoint counts (AES, by country):')
for col in ['post_first_coup','post_french_exit','post_russian_arrival']:
    print(f'\n  {col}:')
    print(pd.crosstab(aes['country'], aes[col]).to_string())

Pre/post breakpoint counts (AES, by country):

  post_first_coup:
post_first_coup  False  True 
country                      
Burkina Faso      4197   7331
Mali              2387   8642
Niger             2957   1463

  post_french_exit:
post_french_exit  False  True 
country                       
Burkina Faso       7162   4366
Mali               5839   5190
Niger              3292   1128

  post_russian_arrival:
post_russian_arrival  False  True 
country                           
Burkina Faso           9256   2272
Mali                   4090   6939
Niger                  3584    836


## 1.10  Export cleaned datasets

In [11]:
out_aes = DATA/'acled_clean.parquet'
out_wa  = DATA/'acled_clean_west_africa.parquet'

aes.to_parquet(out_aes, index=False)
df.to_parquet(out_wa, index=False)

print(f'Wrote {out_aes.relative_to(DATA.parent)}: shape {aes.shape}')
print(f'Wrote {out_wa.relative_to(DATA.parent)} : shape {df.shape}')

Wrote data/acled_clean.parquet: shape (26977, 52)
Wrote data/acled_clean_west_africa.parquet : shape (69598, 49)


## 1.11  Final sanity summary

In [12]:
print('=== AES analysis-ready summary ===')
print(f'Events        : {len(aes):,}')
print(f'Fatalities    : {aes.fatalities.sum():,}')
print(f'Date range    : {aes.event_date.min().date()} → {aes.event_date.max().date()}')
print(f'Countries     : {sorted(aes.country.unique())}')
print()
print('Event type composition (AES):')
print((aes.event_type.value_counts(normalize=True)*100).round(1).astype(str)+'%')
print()
print('actor1_role composition (AES):')
print((aes.actor1_role.value_counts(normalize=True)*100).round(1).astype(str)+'%')
print()
print('Civilian-targeted fatalities by country:')
ct = aes[aes.civilian_targeted].groupby('country')['fatalities'].sum().sort_values(ascending=False)
print(ct.to_string())

=== AES analysis-ready summary ===
Events        : 26,977
Fatalities    : 61,007
Date range    : 2018-01-01 → 2025-04-25
Countries     : ['Burkina Faso', 'Mali', 'Niger']

Event type composition (AES):
event_type
Violence against civilians    31.6%
Battles                       23.5%
Strategic developments        21.9%
Explosions/Remote violence    14.4%
Protests                       6.8%
Riots                          1.7%
Name: proportion, dtype: str

actor1_role composition (AES):
actor1_role
non_state_armed_group    67.9%
state                    19.6%
civilian                  9.9%
external_force            2.6%
Name: proportion, dtype: str

Civilian-targeted fatalities by country:
country
Burkina Faso    10054
Mali             9789
Niger            2901
